In [1]:
import os
import re
import json
import time
from tqdm import tqdm
from collections import Counter

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def create_driver():
    options = uc.ChromeOptions()
    options.add_argument("--lang=en-US")
    options.add_argument("--no-sandbox")
    options.add_argument("--ignore-certificate-errors")
    driver = uc.Chrome(options=options, version_main=149)
    wait = WebDriverWait(driver, 30)
    return driver, wait

def decline_optional_cookies(wait):
    try:
        decline_btn = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(., 'Decline optional cookies')]")
            )
        )
        decline_btn.click()
    except:
        pass

def extract_caption_from_page_source(page_source):
    """Extracts the post caption text from Instagram's embedded JSON."""
    pattern = r'"caption":\{[^}]*?"text":"((?:[^"\\]|\\.)*)"'
    match = re.search(pattern, page_source)
    if not match:
        return ""
    text_raw = match.group(1)
    try:
        text = text_raw.encode('raw_unicode_escape').decode('unicode_escape')
        text = text.encode('utf-16', 'surrogatepass').decode('utf-16')
    except Exception:
        text = text_raw
    return text

def extract_hashtags(caption_text):
    """Extracts all hashtags (without the # symbol) from a caption string."""
    return re.findall(r'#(\w+)', caption_text)

def classify_post(hashtags_found):
    hashtags_lower = {h.lower() for h in hashtags_found}
    has_prolife = bool(hashtags_lower & PROLIFE_HASHTAGS)
    has_prochoice = bool(hashtags_lower & PROCHOICE_HASHTAGS)

    if has_prolife and not has_prochoice:
        return 'pro-life'
    elif has_prochoice and not has_prolife:
        return 'pro-choice'
    elif has_prolife and has_prochoice:
        return 'ambiguous'
    else:
        return 'unclear'

# ============================================================
# CONFIGURATION
# ============================================================
uname = 'auditprojektseminar'
pwd = 'jesuisundummyaccount01'
save_path = 'C:/Users/ondob/Desktop/insta_debug'
urls_file = 'C:/Users/ondob/Desktop/insta_debug/urls.txt'
output_file = f'{save_path}/post_classification.json'

PROLIFE_HASHTAGS = {
    'prolife', 'prolifegeneration', 'endabortion',
    'prolifemovement', 'prolifewomen'
}
PROCHOICE_HASHTAGS = {
    'prochoice', 'mybodymychoice', 'reproductiverights',
    'abortionrights', 'bansoffourbodies'
}

# ============================================================
# LOGIN
# ============================================================
driver, wait = create_driver()
wait_long = WebDriverWait(driver, 30)

driver.get('https://www.instagram.com/')
time.sleep(5)
decline_optional_cookies(wait_long)
time.sleep(2)

try:
    username_input = wait_long.until(EC.presence_of_element_located((By.NAME, "email")))
    username_input.clear()
    username_input.send_keys(uname)
    print("Username OK")
except Exception as e:
    print("Username error:", e)

try:
    password_input = wait_long.until(EC.presence_of_element_located((By.NAME, "pass")))
    password_input.clear()
    password_input.send_keys(pwd)
    print("Password OK")
except Exception as e:
    print("Password error:", e)

try:
    password_input = driver.find_element(By.NAME, "pass")
    password_input.send_keys(Keys.RETURN)
    print("Login submitted OK")
except Exception as e:
    print("Login error:", e)

time.sleep(8)
input("If captcha appears: solve it then press ENTER...")

for _ in range(5):
    try:
        dont_save = wait_long.until(EC.element_to_be_clickable(
            (By.XPATH, "//*[contains(text(), 'Not now') or contains(text(), 'Not Now')]")
        ))
        dont_save.click()
        time.sleep(2)
    except:
        break

cookies = driver.get_cookies()
driver.quit()
print("Login OK\n")

# ============================================================
# CLASSIFICATION DRIVER
# ============================================================
driver, wait = create_driver()
wait_long = WebDriverWait(driver, 30)

driver.get('https://www.instagram.com/')
time.sleep(4)
for cookie in cookies:
    try:
        driver.add_cookie(cookie)
    except:
        pass
driver.refresh()
time.sleep(5)
print("Driver ready\n")

# ============================================================
# LOAD URL LIST
# ============================================================
with open(urls_file, 'r') as f:
    post_urls = [line.strip() for line in f if line.strip()]

print(f"URLs to classify: {len(post_urls)}")

results = {}
if os.path.exists(output_file):
    try:
        with open(output_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
    except (json.JSONDecodeError, ValueError):
        print(f" Warning: {output_file} was corrupted. Starting with fresh/empty results.")
        results = {}

already_done = set(results.keys())

# ============================================================
# CLASSIFY EACH POST
# ============================================================
for url in tqdm(post_urls):
    url_id = url.rstrip('/').split('/')[-1]

    if url_id in already_done:
        continue

    driver.get(url)
    time.sleep(4)
    decline_optional_cookies(wait_long)

    page_source = driver.page_source
    caption = extract_caption_from_page_source(page_source)
    hashtags = extract_hashtags(caption)
    classification = classify_post(hashtags)

    results[url_id] = {
        "url": url,
        "hashtags_found": hashtags,
        "classification": classification
    }

    # Atomic Save: Write to temp file then replace original to avoid corruption
    temp_file = f"{output_file}.tmp"
    with open(temp_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    os.replace(temp_file, output_file)

    print(f"{url_id}: {classification} (hashtags: {hashtags[:5]})")
    time.sleep(2)

driver.quit()

# ============================================================
# SUMMARY
# ============================================================
counts = Counter(v['classification'] for v in results.values())
print(f"\n{'='*50}")
print("CLASSIFICATION SUMMARY")
print(f"{'='*50}")
for label, count in counts.items():
    print(f"  {label}: {count}")
print(f"\nSaved to: {output_file}")

# ============================================================
# UNCLEAR POSTS - HASHTAG FREQUENCY ANALYSIS
# ============================================================
unclear_posts = {
    url_id: data for url_id, data in results.items()
    if data['classification'] == 'unclear'
}

print(f"\n{'='*50}")
print(f"UNCLEAR POSTS ANALYSIS ({len(unclear_posts)} posts)")
print(f"{'='*50}")

hashtag_counter = Counter()
for data in unclear_posts.values():
    hashtags_lower = [h.lower() for h in data['hashtags_found']]
    hashtag_counter.update(hashtags_lower)

print("\nMost common hashtags in UNCLEAR posts (candidates to add):\n")
for hashtag, count in hashtag_counter.most_common(40):
    print(f"  #{hashtag}: {count} occurrences")

print(f"\n{'='*50}")
print("Sample of 5 unclear posts with their full hashtag lists:")
print(f"{'='*50}")
for i, (url_id, data) in enumerate(unclear_posts.items()):
    if i >= 5:
        break
    print(f"\n{data['url']}")
    print(f"  Hashtags: {data['hashtags_found']}")

Username OK
Password OK
Login submitted OK
Login OK

Driver ready

URLs to classify: 664


  0%|          | 0/664 [00:00<?, ?it/s]

DZht7Y3lhPh: unclear (hashtags: ['abortion', 'testimony', 'mytestimony', 'downsyndrome', 'trisomy21'])


 96%|█████████▌| 635/664 [01:00<00:01, 14.83it/s]

BuSmbpsn7g4: unclear (hashtags: [])


...

DTsRYHhiLBC: unclear (hashtags: ['abortion', 'atheism', 'catholicism'])


100%|█████████▉| 663/664 [20:02<00:39, 39.31s/it]

DSHVXVUkiEr: unclear (hashtags: ['drgampalasirisha', 'saffron', 'saffron', 'pregnancy', 'babycolour'])


100%|██████████| 664/664 [20:40<00:00,  1.87s/it]



CLASSIFICATION SUMMARY
  unclear: 302
  ambiguous: 74
  pro-life: 189
  pro-choice: 99

Saved to: C:/Users/ondob/Desktop/insta_debug/post_classification.json

UNCLEAR POSTS ANALYSIS (302 posts)

Most common hashtags in UNCLEAR posts (candidates to add):

  #abortion: 186 occurrences
  #fyp: 17 occurrences
  #viral: 15 occurrences
  #christian: 15 occurrences
  #pregnancy: 14 occurrences
  #jesus: 14 occurrences
  #downsyndrome: 10 occurrences
  #politics: 10 occurrences
  #abortionishealthcare: 9 occurrences
  #women: 8 occurrences
  #faith: 7 occurrences
  #debate: 7 occurrences
  #explore: 7 occurrences
  #healthcare: 7 occurrences
  #trending: 7 occurrences
  #christianity: 6 occurrences
  #news: 6 occurrences
  #shoutyourabortion: 6 occurrences
  #god: 6 occurrences
  #abortionisfreedom: 6 occurrences
  #miscarriage: 6 occurrences
  #feminism: 6 occurrences
  #religion: 6 occurrences
  #abortions: 5 occurrences
  #secularprolife: 5 occurrences
  #truth: 5 occurrences
  #catholic: 

In [6]:
import json
from collections import Counter

output_file = 'C:/Users/ondob/Desktop/insta_debug/post_classification.json'

PROLIFE_HASHTAGS = {
    'prolife', 'prolifegeneration', 'endabortion',
    'prolifemovement', 'prolifewomen',
    'secularprolife', 'abortionismurder', 'downsyndrome',
    'christian', 'christianity', 'catholic', 'catholicism',
    'jesus', 'faith', 'god', 'religion', 'baby',
    'conservative', 'republicans'
}

PROCHOICE_HASHTAGS = {
    'prochoice', 'mybodymychoice', 'reproductiverights',
    'abortionrights', 'bansoffourbodies',
    'shoutyourabortion', 'abortionisfreedom', 'abortionishealthcare',
    'reproductivejustice', 'feminism', 'healthcare'
}


def classify_post(hashtags_found):
    hashtags_lower = {h.lower() for h in hashtags_found}
    has_prolife = bool(hashtags_lower & PROLIFE_HASHTAGS)
    has_prochoice = bool(hashtags_lower & PROCHOICE_HASHTAGS)

    if has_prolife and not has_prochoice:
        return 'pro-life'
    elif has_prochoice and not has_prolife:
        return 'pro-choice'
    elif has_prolife and has_prochoice:
        return 'ambiguous'
    else:
        return 'unclear'


# ============================================================
# LOAD EXISTING RESULTS - explicit UTF-8 to avoid Windows cp1252 issues
# ============================================================
with open(output_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

before_counts = Counter(v['classification'] for v in results.values())
print("BEFORE re-classification:")
for label, count in before_counts.items():
    print(f"  {label}: {count}")

# ============================================================
# RE-CLASSIFY ONLY POSTS CURRENTLY MARKED 'unclear'
# ============================================================
reclassified = 0

for url_id, data in results.items():
    if data['classification'] != 'unclear':
        continue

    new_classification = classify_post(data['hashtags_found'])

    if new_classification != 'unclear':
        data['classification'] = new_classification
        reclassified += 1

# ============================================================
# SAVE UPDATED RESULTS - explicit UTF-8 again
# ============================================================
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n{reclassified} posts moved out of 'unclear'")

after_counts = Counter(v['classification'] for v in results.values())
print("\nAFTER re-classification:")
for label, count in after_counts.items():
    print(f"  {label}: {count}")

print(f"\nSaved to: {output_file}")

BEFORE re-classification:
  unclear: 211
  pro-life: 247
  ambiguous: 79
  pro-choice: 127

0 posts moved out of 'unclear'

AFTER re-classification:
  unclear: 211
  pro-life: 247
  ambiguous: 79
  pro-choice: 127

Saved to: C:/Users/ondob/Desktop/insta_debug/post_classification.json


In [8]:
import json
from collections import Counter

output_file = 'C:/Users/ondob/Desktop/insta_debug/post_classification.json'

# ============================================================
# LOAD EXISTING RESULTS
# ============================================================
with open(output_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

before_counts = Counter(v['classification'] for v in results.values())
print("BEFORE relabeling:")
for label, count in before_counts.items():
    print(f"  {label}: {count}")

# ============================================================
# RELABEL ALL REMAINING 'unclear' POSTS AS 'abortion'
# ============================================================
relabeled = 0

for url_id, data in results.items():
    if data['classification'] == "unclear":
        data['classification'] = "abortion"
        relabeled += 1

# ============================================================
# SAVE UPDATED RESULTS
# ============================================================
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n{relabeled} posts relabeled from 'unclear' to 'abortion'")

after_counts = Counter(v['classification'] for v in results.values())
print("\nAFTER relabeling (final 4 categories):")
for label, count in after_counts.items():
    print(f"  {label}: {count}")

print(f"\nSaved to: {output_file}")

BEFORE relabeling:
  abortion: 211
  pro-life: 247
  ambiguous: 79
  pro-choice: 127

0 posts relabeled from 'unclear' to 'abortion'

AFTER relabeling (final 4 categories):
  abortion: 211
  pro-life: 247
  ambiguous: 79
  pro-choice: 127

Saved to: C:/Users/ondob/Desktop/insta_debug/post_classification.json


In [8]:
import json
import os

output_file = 'C:/Users/ondob/Desktop/insta_debug/post_classification.json'

# 1. Confirme que le fichier existe bien à cet endroit
print("File exists:", os.path.exists(output_file))
print("File size:", os.path.getsize(output_file), "bytes")
print("Last modified:", os.path.getmtime(output_file))

# 2. Charge et inspecte les valeurs de classification réelles
with open(output_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"\nTotal posts: {len(results)}")

# 3. Regarde les valeurs UNIQUES de classification (avec repr pour voir espaces/casse cachés)
unique_classifications = set(v['classification'] for v in results.values())
print("\nUnique classification values found (repr to catch hidden whitespace/case):")
for c in unique_classifications:
    print(f"  {repr(c)}")

File exists: True
File size: 181432 bytes
Last modified: 1784753874.4891644

Total posts: 664

Unique classification values found (repr to catch hidden whitespace/case):
  'pro-choice'
  'abortion'
  'pro-life'
  'ambiguous'
